# IHLT Project
Students : Oscar Cubeles, Valentin-Gabriel Soumah

Description of the Project and Notebook ...

In [14]:
# Base Libraries
import os
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# NLP Libraries
import spacy
from scipy.stats import pearsonr

# Import Our Python Modules
import features as ft
import preprocessing as pp
import feature_selection as fs
from load_data import load_files, merge_datasets 

# ML Libraries
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.inspection import permutation_importance

importlib.reload(ft)
importlib.reload(pp)

nlp = spacy.load("en_core_web_md")

## 1. Data Preparation

### 1.1 Loading Data

In [5]:
data_path = 'data/train/train/'
df_paraphrases = load_files(data_path+'STS.gs.MSRpar.txt', data_path+'STS.input.MSRpar.txt')
df_vid = load_files(data_path+'STS.gs.MSRvid.txt', data_path+'STS.input.MSRvid.txt')
df_europarl = load_files(data_path+'STS.gs.SMTeuroparl.txt', data_path+'STS.input.SMTeuroparl.txt') 

In [6]:
df = merge_datasets([df_paraphrases, df_vid , df_europarl])
df.head()

,sentence1,sentence2,score
0,But other sources close to the sale said Viven...,But other sources close to the sale said Viven...,4.00
1,Micron has declared its first quarterly profit...,Micron's numbers also marked the first quarter...,3.75
2,The fines are part of failed Republican effort...,"Perry said he backs the Senate's efforts, incl...",2.80
3,"The American Anglican Council, which represent...","The American Anglican Council, which represent...",3.40
4,The tech-loaded Nasdaq composite rose 20.96 po...,The technology-laced Nasdaq Composite Index <....,2.40


### 1.2 Preprocessing

Before computing the features, we do some simple preprocessing steps to improve the similarity computation. It is important to note that here we do not remove stopwords as they are meaningful for the pos taggers, thus removal of stopwords is computed on features that require so. 

1. All hyphens "-", slashes "/" and angular brackets (< and >) are removed;
2. The currency values are simplified, e.g., $US1234 to $1234 (simple);
3. Replace contractions. e.g: n’t with not, ’m with am


In [7]:
df_preprocessed = pp.preprocess_dataframe(df)
df_preprocessed.head(15)
print(df_preprocessed.iloc[11]["sentence1"])

Replacements were made:
Before: Chavez said investigators feel confident they've got "at least one of the fires resolved in that regard."
After:  Chavez said investigators feel confident they have got "at least one of the fires resolved in that regard."
Replacements were made:
Before: Plofsky said the commission won't investigate because the three year statute of limitations has expired.
After:  Plofsky said the commission will not investigate because the three year statute of limitations has expired.
Replacements were made:
Before: In two weeks, he'll probably send out Peace Rules in the Preakness.
After:  In two weeks, he will probably send out Peace Rules in the Preakness.
Replacements were made:
Before: Still, he said, "I'm absolutely confident we're going to have a bill."
After:  Still, he said, "I am absolutely confident we are going to have a bill."
Replacements were made:
Before: "I'm absolutely confident we're going to have a bill," Frist, R Tenn., said Thursday.
After:  "I am

### 1.3 Generating Features

In [8]:
match_based_features = [ 
            ft.word_jaccard_distance, 
            ft.compute_number_similarity,
            ft.date_similarity,
            ft.word_bigram_jaccard, 
            ft.word_trigram_jaccard,
]

lexical_features = [ft.synset_jaccard_distance,
            ft.weighted_wordnet_similarity,
            ft.sentence_path_similarity,
            ft.sentence_lch_similarity,
            ft.sentence_wup_similarity,
            ft.sentence_lin_similarity,
            ft.sentence_path_similarity_max,
            ft.sentence_lch_similarity_max,
            ft.sentence_wup_similarity_max,
            ft.sentence_lin_similarity_max,
            ft.sentence_rarity_similarity,
            ft.stem_jaccard_distance,
            ft.lemma_jaccard_distance,
            ft.lemma_bigram_jaccard, 
            ft.lemma_trigram_jaccard
]

syntactic_features = [ft.pos_jaccard_distance,
            ft.sentence_length_difference,
            ft.longest_common_subsequence, 
            ft.longest_common_substring,
            ft.greedy_string_tiling,
            ft.pos_lev_distance,
            ft.dep_lev_distance,
            ft.ner_jaccard,
            ft.ner_presence,
]

match_based_features_names = [func.__name__ for func in match_based_features]
lexical_match_features_names = [func.__name__ for func in lexical_features] + match_based_features_names
syntactic_match_features_names = [func.__name__ for func in syntactic_features] + match_based_features_names
features = match_based_features  + lexical_features + syntactic_features

From all the features that we have designed, those have been divided into 3 categories: 
- **Match-based features**: Those features do not rely on any external resources nor annotations, they are computed purely based on the immediate form of the sentences and their tokens. 

     - **Word Unigram distance, Bigram and Trigram Jaccard Similarity**: Measures sentences comparing the overlap of word sequences using Jaccard distance for unigrams and similarity for bigrams and trigrams. For example, unigrams (`["alex", "played"]`), bigrams (`["alex played"]`), and trigrams (`["alex played football"]`) these freatures capture shared words and short sequences of words between sentences.
     - **Number Similarity**: Computes number similarity even if numbers are in word format. Also, computing with dice similarity to give more importance to intersections (reward more small overlaps)
     - **Date Similarity**: Finds all potential date objects in both sentences and computes the similarity based on the time distance between dates. E.g: Same date returns 1




- **Lexical Features**: Those features leverage external lexical ressources (wordnet or more basic lexicons for lemmatization)
     - **Synset Jaccard Distance**:  Measures sentence similarity based on the overlap of word senses (synsets) derived using WSD. First uses the Lesk algorithm to assign synsets to words, then computes the Jaccard similarity between the sets of synsets for each sentence. 
     - **Weighted WordNet Similarity** and **Sentence Rarity Similarity**: Features that measure sentence similarity by comparing token overlap, with weights based on word rarity (from the Brown corpus) and semantic depth in WordNet. Rare and semantically specific words contribute more to the similarity score by having high weights.
     - **Lemma Unigram Distance, Bigram and Trigram Jaccard Similarity**: Measures sentence similarity by comparing the overlap of word sequences after lemmatizing the words in the sentences. Similar to the word-based unigram, bigram, and trigram comparisons, but instead of raw words, the words are reduced to their base forms (lemmas). For example, after lemmatizing, `["playing"]` and `["played"]` both become `["play"]`, allowing for comparison of shared lemmas.

    - **Sentence Path, LCH, WUP, and Lin Similarity**: These features compute semantic similarity between sentences using different WordNet metrics:
      - **Path**: Path-based similarity
      - **LCH**: Leacock-Chodorow similarity
      - **WUP**: Wu-Palmer similarity
      - **Lin**: Lin similarity based on information content

        - **Aggregation (Mean vs Max)**:
          - **`mean`**: Default, which averages the pairwise similarity scores between words in both sentences.
          - **`max`**: Selects the highest similarity score between any pair of words.






     




### Lexical Features

In [ ]:
sentence1 = nlp("I love playing very competitive football")
sentence2 = nlp("She was playing football today")


print(f"{ft.synset_jaccard_distance(sentence1, sentence2)}")
print(f"{ft.weighted_wordnet_similarity(sentence1, sentence2)}")
print(f"{ft.sentence_path_similarity(sentence1, sentence2)}")
print(f"{ft.sentence_lch_similarity(sentence1, sentence2)}")
print(f"{ft.sentence_wup_similarity(sentence1, sentence2)}")
print(f"{ft.sentence_lin_similarity(sentence1, sentence2)}")
print(f"{ft.sentence_path_similarity_max(sentence1, sentence2)}")
print(f"{ft.sentence_lch_similarity_max(sentence1, sentence2)}")
print(f"{ft.sentence_wup_similarity_max(sentence1, sentence2)}")
print(f"{ft.sentence_lin_similarity_max(sentence1, sentence2)}")
print(f"{ft.sentence_rarity_similarity(sentence1, sentence2)}")
print(f"{ft.stem_jaccard_distance(sentence1, sentence2)}")
print(f"{ft.lemma_jaccard_distance(sentence1, sentence2)}")
print(f"{ft.lemma_bigram_jaccard(sentence1, sentence2)}") 
print(f"{ft.lemma_trigram_jaccard(sentence1, sentence2)}")


- **Syntactic features**: Those feature harness the annotations made by NLP annotation tools (parsing, pos-tagging etc). In our case, all the annotations are provided by spacy NLP pipeline.
     - **Sentence Length similarity**: Absolute difference in sentence lengths divided by the maximum length. We considered syntactical because it reflects a sentence’s structure and complexity, longer sentences often have more clauses or phrases.
     
     - **POS Jaccard Distance and POS Levenshtein Distance**: Measure similarity between sentences based on **POS tags**. Jaccard distance considers overlap of POS tags as a set, while Levenshtein distance measures the minimum edits needed to transform one POS sequence into another.  
     - **Dependency Label Distance**: Computes the Levenshtein distance between sequences of dependency labels (`nsubj`, `obj`, `ROOT`, etc.), measuring the minimum edits to transform one sequence into another and capturing syntactic structure differences.
     - **Longest Common Subsequence (LCS) Similarity**: Measures sentence similarity by finding the longest sequence of words that appear in order in both sentences (not necessarily contiguous) and normalizing by the length of the longer sentence. For example, between `"I love playing very competitive football"` and `"She was playing football today"`, the LCS `"playing football"` is detected, giving a normalized similarity of 0.333.  

     - **Longest Common Substring (LCSubstr) Similarity**: Measures similarity by finding the longest contiguous sequence of words that appear in the same order in both sentences, normalized by the length of the longer sentence. For instance, between `"I love playing very competitive football"` and `"She was playing football today"`, the contiguous substring `"playing"` is the longest match.


     - **Greedy String Tiling (GST)**: Measures sentence similarity by finding the longest non-overlapping matching token sequences of at least a minimum length, marking them as used. All matched tiles contribute to the similarity score, which is normalized by the total number of tokens. For example, between `"I love playing very competitive football"` and `"She was playing football today"` with `min_match_length=1`, the matches `"playing"` and `"football"` are identified..


     
     - **Named Entity Presence and Named Entity Distance**:  
        - **Presence**: Binary feature indicating whether either sentence contains named entities (e.g., `PERSON`, `ORG`, `DATE`).  
        - **Distance**: Measures similarity of named entity types between sentences using Jaccard distance. Considered syntactic/semantic because named entities are often associated with proper nouns and sentence structure patterns.





TODO: Describe..

### Syntactical Features

In [19]:
sentence1 = nlp("I love playing very competitive football")
sentence2 = nlp("She was playing football today")

print(f" longest common subsequence: {ft.longest_common_subsequence(sentence1, sentence2)}")
print(f" longest common substring: {ft.longest_common_substring(sentence1, sentence2)}")
print(f" greedy string tiling: {ft.greedy_string_tiling(sentence1, sentence2, min_match_length=1)}")


Longest Common Subsequence words (in order, non-contiguous): ['playing', 'football']
LCS length: 2, Normalized similarity: 0.333
 longest common subsequence: 0.3333333333333333
Longest Common Substring Words: ['playing']
 longest common substring: 0.16666666666666666
Found match: ['playing'] (length 1) at positions S1[2:3], S2[2:3]
Found match: ['football'] (length 1) at positions S1[5:6], S2[3:4]
Total matched tokens: 2, similarity: 0.3636
 greedy string tiling: 0.36363636363636365


In [9]:
sentence1 = nlp("Today Alex was playing football in the park with 5 or 6 friends.")
sentence2 = nlp("In the park, Alex and six friends played football today.")




print(f" sentence length difference: {ft.sentence_length_difference(sentence1, sentence2)}")
print(f" pos jaccard distance: {ft.pos_jaccard_distance(sentence1, sentence2)}")
print(f" longest common subsequence: {ft.longest_common_subsequence(sentence1, sentence2)}")
print(f" longest common substring: {ft.longest_common_substring(sentence1, sentence2)}")
print(f" greedy string tiling: {ft.greedy_string_tiling(sentence1, sentence2)}")
print(f" pos lev distance: {ft.pos_lev_distance(sentence1, sentence2)}")
print(f" dep lev distance: {ft.dep_lev_distance(sentence1, sentence2)}")
print(f" ner jaccard: {ft.ner_jaccard(sentence1, sentence2)}")
print(f" ner presence: {ft.ner_presence(sentence1, sentence2)}")

 sentence length difference: 0.23076923076923078
Tokens S1: {'n', 'a', 'v'}
Tokens S2: {'n', 'a', 'v'}
 pos jaccard distance: 0.0
 longest common subsequence: 0.15384615384615385
 longest common substring: 0.07692307692307693
 greedy string tiling: 0.0
 pos lev distance: 10
 dep lev distance: 11
 ner jaccard: 0.0
 ner presence: 1


### Match Based Features

In [31]:
sentence1 = nlp("Today Alex was playing football in the park with 5 or 6 friends.")
sentence2 = nlp("In the park, Alex and six friends played football today.")

print("Feature values for the example sentences:")
print(f" Jaccard distance: {ft.word_jaccard_distance(sentence1, sentence2)}")
print(f" Number similarity: {ft.compute_number_similarity(sentence1, sentence2)}")
print(f" Date similarity: {ft.date_similarity(sentence1, sentence2)}")
print(f" Bigram Jaccard: {ft.word_bigram_jaccard(sentence1, sentence2)}")
print(f" Trigram Jaccard: {ft.word_trigram_jaccard(sentence1, sentence2)}")

sentence3 = nlp("The meeting is scheduled for 12/05/2023.")
sentence4 = nlp("We will meet on the 12th of May 2023.")

print("\nFeature values for the date example sentences:")
print(f" Jaccard distance: {ft.word_jaccard_distance(sentence3, sentence4)}")
print(f" Number similarity: {ft.compute_number_similarity(sentence3, sentence4)}")  
print(f" Date similarity: {ft.date_similarity(sentence3, sentence4)}")
print(f" Bigram Jaccard: {ft.word_bigram_jaccard(sentence3, sentence4)}")
print(f" Trigram Jaccard: {ft.word_trigram_jaccard(sentence3, sentence4)}")



Feature values for the example sentences:
Tokens S1: {'friends', 'playing', 'today', '5', 'park', 'alex', '6', 'football'}
Tokens S2: {'friends', 'today', 'played', 'park', 'alex', 'football'}
 Jaccard distance: 0.4444444444444444
Numbers S1: {5.0, 6.0}
Numbers S2: {6.0}
 Number similarity: 0.6666666666666666
 Date similarity: 0.0
 Bigram Jaccard: 0.09090909090909091
 Trigram Jaccard: 0.0

Feature values for the date example sentences:
Tokens S1: {'scheduled', 'meeting', '12/05/2023'}
Tokens S2: {'2023', 'meet', '12th'}
 Jaccard distance: 1.0
Numbers S1: {12.0, 5.0, 2023.0}
Numbers S2: {12.0, 2023.0}
 Number similarity: 0.8
Best pair: (datetime.date(2023, 5, 12), datetime.date(2023, 5, 1)), Delta: 11 days, Similarity: 0.8479
 Date similarity: 0.8478937040879159
 Bigram Jaccard: 0.0
 Trigram Jaccard: 0.0
